# 04 — Results, tables and figures

Everything the final report states, recomputed from the committed result files: the server
results against the published systems, the held-out tests on VisA and MPDD, the AD 2
development tables, and the eight figures. No dataset and no GPU are needed. Report:
`reports/00-final-report.md`.

The figures plot metrics only, never dataset imagery. Data behind the numbers: VisA © Amazon,
CC BY 4.0; MPDD and MVTec AD 2, CC BY-NC-SA 4.0. See [ATTRIBUTION.md](../ATTRIBUTION.md).

## 0. Environment

Needs only the `notebook` extra (pandas, matplotlib).

In [ ]:
import os, sys, json, pathlib, subprocess

REPO = pathlib.Path.cwd()
if not (REPO / 'src' / 'inspector').exists():
    REPO = REPO.parent  # running from notebooks/
os.chdir(REPO)  # scripts and configs are addressed from the repository root
sys.path.insert(0, str(REPO / 'src'))

from inspector.utils.env import capture

env = capture()
print('git      :', env['git_sha'], '(dirty)' if env['dirty'] else '(clean)')
print('device   :', env['device'].get('gpu_name', 'CPU only'),
      f"{env['device'].get('gpu_total_mb', 0)} MB")

def sh(*args, run=True):
    """Run a repository script (or `-m inspector ...`) with this interpreter.
    The command is always printed; with run=False it is only printed."""
    cmd = [sys.executable, *map(str, args)]
    print('$ python', ' '.join(cmd[1:]), '' if run else '   (not run)')
    if run:
        subprocess.run(cmd, check=True)

In [ ]:
import pandas as pd
from IPython.display import Image, Markdown, display

STUDIES = REPO / 'reports' / 'studies'
pd.set_option('display.precision', 3)

## 1. MVTec's server: the two submissions and the published systems

Means over the eight categories, `test_private` (training lighting) and `test_private_mixed`
(changed lighting). v6 met the pre-registered H5 bar on both splits.

In [ ]:
server = pd.read_csv(STUDIES / 'ad2-server' / 'server_means.csv')
display(server.pivot(index='recipe', columns='split', values=['aupro_005', 'segf1', 'classf1']))
published = pd.read_csv(STUDIES / 'ad2-server' / 'published_reference.csv')
display(published.drop(columns=['source']).sort_values('segf1_mixed', ascending=False))

## 2. Held-out tests of the frozen v1 recipe: VisA (E12) and MPDD (H1)

In [ ]:
visa = pd.DataFrame(json.loads((STUDIES / 'confirmation-v1' / 'confirmation.json').read_text(encoding='utf-8'))['descriptive'])
display(visa[['category', 'auroc_640', 'aupro005_640', 'auroc_320', 'auroc_pixel_pca']])
mpdd = pd.read_csv(STUDIES / 'kaggle' / 'mpdd.csv')
display(mpdd[['category', 'image_auroc_mean', 'aupro_005_mean', 'pixel_pca_auroc', 'fp_total', 'fp_expected']])

## 3. MVTec AD 2 development: v5's rule against v6's head, per category

`test_public`, seed 0. `regular` captures approximate `test_private`, all captures
`test_private_mixed`. The intervals resample objects within categories.

In [ ]:
check = json.loads((STUDIES / 'fusion-head' / 'hybrid_check.json').read_text(encoding='utf-8'))
boot = json.loads((STUDIES / 'fusion-head' / 'hybrid_bootstrap.json').read_text(encoding='utf-8'))
rows = []
for cat, arms in check['categories'].items():
    for subset in ('regular', 'all'):
        b = boot['categories'][cat][subset]
        rows.append({'category': cat, 'subset': subset, 'v5 rule': arms['rule'][subset],
                     'v6 head': arms['hybrid'][subset], 'gain': b['diff'],
                     'ci low': b['ci95'][0], 'ci high': b['ci95'][1]})
display(pd.DataFrame(rows))
display(Markdown(f"macro AU-PRO, regular / all: v5 {check['macro']['rule']['regular']:.3f} / "
                 f"{check['macro']['rule']['all']:.3f}, v6 {check['macro']['hybrid']['regular']:.3f} / "
                 f"{check['macro']['hybrid']['all']:.3f}"))

## 4. The figures of the final report

In [ ]:
sh('scripts/report_figures.py')
for png in sorted((REPO / 'reports' / 'figures').glob('fig*.png')):
    display(Markdown(f'**{png.stem}**'))
    display(Image(filename=str(png)))

## 5. The run registry

Every run of the project, one row each, with its implementation id and git SHA.

In [ ]:
registry = pd.read_csv(REPO / 'reports' / 'results_registry.csv')
display(registry.groupby(['dataset', 'method']).size().rename('runs').to_frame())